In [1]:
from pathlib import Path
import importlib
import os
import uproot
import polars as pl
import plotly.graph_objects as go
import qmirt_utility as qmirt
import numpy as np

importlib.reload(qmirt)

<module 'qmirt_utility' from '/home/fanghan/Work/RPIL/QMIRT/gate10mc/dev/python/qmirt-utility/qmirt_utility/__init__.py'>

In [2]:
output_dir = os.path.abspath(os.path.join(os.getcwd(), "../../output"))
root_filename = "box_lyso_sipm_output.root"
with uproot.open(Path(output_dir) / root_filename) as file:
    print(file.keys())

['crystal_singles;1', 'opt_epoxy_phsa;1']


In [3]:
output_dir = os.path.abspath(os.path.join(os.getcwd(), "../../output"))
root_filename = "box_lyso_sipm_output.root"
with uproot.open(Path(output_dir) / root_filename) as file:
    tree = file["crystal_singles"]
    crystal_singles_df =  pl.DataFrame(tree.arrays(tree.keys(), library="np"))
    tree = file["opt_epoxy_phsa"]
    opt_epoxy_phsa_df = pl.DataFrame(tree.arrays(tree.keys(), library="np"))
print(crystal_singles_df.shape)
print(opt_epoxy_phsa_df.shape)
print(crystal_singles_df.columns)

(11, 12)
(52993, 13)
['EventID', 'TotalEnergyDeposit', 'KineticEnergy', 'PostPosition_X', 'PostPosition_Y', 'PostPosition_Z', 'PrePosition_X', 'PrePosition_Y', 'PrePosition_Z', 'PreStepUniqueVolumeID', 'ParticleName', 'GlobalTime']


In [4]:
style_sheet_path = "wrl_stylesheet.example.json"
fig = qmirt.plot.plot_wrl_file(
    "../../box_lyso_sipm.wrl",
    style_sheet=style_sheet_path,
    exclude_patterns=["world:*"],
    # exclude_patterns=["crystal:*", "world:*"],
    exclude_mode="replace",
)

singles_positions = crystal_singles_df[
    ["PostPosition_X", "PostPosition_Y", "PostPosition_Z"]
].to_numpy()
total_deposited_energy = crystal_singles_df["TotalEnergyDeposit"].to_numpy()*1.0e3 # convert from MeV to keV
event_ids = crystal_singles_df["EventID"]

singles_traces = []
for single_position, event_id, energy in zip(singles_positions, event_ids, total_deposited_energy):
    singles_traces.append(
        go.Scatter3d(
            x=[single_position[0]],
            y=[single_position[1]],
            z=[single_position[2]],
            mode="markers",
            marker=dict(
                size=2,
                color="red",
                opacity=0.8,
            ),
            name=f"Event ID: {event_id}; Energy: {energy:.2f} keV",
        )
    )
fig.add_traces(singles_traces)

fig.show()

In [5]:
## Get ParticleNams in epoxy_phsa
print(opt_epoxy_phsa_df["ParticleName"].unique())
# Filter optical photons in epoxy_phsa
optical_photons_df = opt_epoxy_phsa_df.filter(pl.col("ParticleName") != "opticalphoton")
print(optical_photons_df.shape)
# print(optical_photons_df.columns)


shape: (1,)
Series: 'ParticleName' [str]
[
	"opticalphoton"
]
(0, 13)
